<a href="https://colab.research.google.com/github/henry04012006/Matryoshka-Adaptor/blob/main/Matryoshka_Adaptor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Implementing the unsupervised training of Matryoshka-Adaptors <br/>
With the currently set parameters, the code trains on the SciFact corpus using the unsupervised objectve. <br/>
Afterwards, the adaptor's retrieval performance is evaluated on the SciFact test dataset. <br/>
Note: You can instead train on MSMARCO by setting: tr_dataset_name = "msmarco"

In [ ]:
!pip install sentence-transformers beir mteb faiss-cpu

  Using cached beir-2.2.0-py3-none-any.whl.metadata (28 kB)
  Using cached mteb-2.10.3-py3-none-any.whl.metadata (16 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.1/5.1 MB 65.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 71.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 35.4 MB/s eta 0:00:00


In [ ]:
import copy
from beir import util, LoggingHandler
from beir.datasets.data_loader import GenericDataLoader
from sentence_transformers import SentenceTransformer

import torch
import torch.optim as optim
from torch import nn

import numpy as np
import os

import faiss
import random
import mteb
from mteb import MTEB
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt

from typing import Dict, Union

/usr/local/lib/python3.12/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


In [ ]:
# setting the random seed to improve reproducibility
random_seed = 0

python_random = random.Random(random_seed)
torch.manual_seed(random_seed)

### Hyperparameters

In [ ]:
# Most hyperparameters are copied from Appendix B of the paper

base_encoder = "sentence-transformers/all-MiniLM-L6-v2"
embedding_dim = 384
m_values = [64, 128, 256, 384]

# Matryoshka Adaptor hyperparameters
ma_hidden_dim = 1024

# Training/eval dataset profile for Colab (default: moderate size, not tiny debug)
tr_dataset_name = "msmarco"
eval_dataset_name = "scifact"

# Colab-friendly corpus cap: keeps training signal meaningful while avoiding OOM/timeouts
MAX_DOCS = 50000

# Optional debug mode (only for quick smoke tests when resource errors happen)
DEBUG_SMALL_RUN = False
DEBUG_MAX_DOCS = 5000
max_num_tr_datapoints = DEBUG_MAX_DOCS if DEBUG_SMALL_RUN else MAX_DOCS

k = 5  # number of nearest neighbours used in Eq.2 top-k loss
batch_size = 128
optimizer_fn = lambda params: optim.Adam(params, lr=0.001)
patience = 500
max_tr_iterations = 5500  # light tuning: a few more steps for more stable sanity-check
log_every = 50
grad_clip_norm = 1.0

# Eq.4 weights (paper-style): keep defaults at 1.0, but expose for light tuning later.
LAMBDA_REC = 1.0
LAMBDA_PAIR = 1.0
LAMBDA_KNN = 1.0

# Optional light tuning profile for Colab stability (small change, no major refactor).
USE_LIGHT_TUNING = True
TRAIN_LAMBDA_REC = LAMBDA_REC
TRAIN_LAMBDA_PAIR = 0.9 if USE_LIGHT_TUNING else LAMBDA_PAIR
TRAIN_LAMBDA_KNN = 1.1 if USE_LIGHT_TUNING else LAMBDA_KNN

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Runtime device: {device}, train dataset: {tr_dataset_name}, eval dataset: {eval_dataset_name}, max docs: {max_num_tr_datapoints}")
print(f"Training config -> m_values={m_values}, k={k}, batch_size={batch_size}, lambdas(default)=({LAMBDA_REC},{LAMBDA_PAIR},{LAMBDA_KNN}), train=({TRAIN_LAMBDA_REC},{TRAIN_LAMBDA_PAIR},{TRAIN_LAMBDA_KNN})")


### Adaptor Model

In [ ]:
class SwiGLU(nn.Module):
    def __init__(self, dim_in, dim_out):
        super(SwiGLU, self).__init__()
        self.linear_1 = nn.Linear(dim_in, dim_out)
        self.linear_2 = nn.Linear(dim_in, dim_out)

    def forward(self, x):
        # First linear projection followed by SiLU (Sigmoid Linear Unit)
        x1 = torch.nn.functional.silu(self.linear_1(x))
        # Second linear projection
        x2 = self.linear_2(x)
        # Element-wise multiplication
        return x1 * x2

class MatryoshkaAdaptor(nn.Module):
    def __init__(self, embedding_dim, hidden_dim):
        super(MatryoshkaAdaptor, self).__init__()
        self.layer1 = SwiGLU(embedding_dim, hidden_dim)
        self.layer2 = nn.Linear(hidden_dim, embedding_dim)

        # Start exactly from identity mapping (via skip connection) to avoid early quality drop.
        nn.init.zeros_(self.layer2.weight)
        nn.init.zeros_(self.layer2.bias)

    def forward(self, x: Union[torch.Tensor, Dict[str, torch.Tensor]]):
        if isinstance(x, Dict):
            inputs = x["sentence_embedding"]
        else:
            inputs = x

        out = self.layer1(inputs)
        out = self.layer2(out)
        out = out + inputs  # Skip connection

        if isinstance(x, Dict):
            x["sentence_embedding"] = out
            return x
        else:
            return out

## Data Prep Functions

In [ ]:
def load_and_preprocess_data(dataset_name, model_name, batch_size=512):
    # 1) Load the BEIR dataset
    url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset_name}.zip"
    data_path = util.download_and_unzip(url, f"./datasets/{dataset_name}")
    corpus, queries, qrels = GenericDataLoader(data_folder=data_path).load(split="train")

    corpus_keys = list(corpus.keys())
    if len(corpus) > max_num_tr_datapoints:
        corpus_keys = python_random.sample(corpus_keys, max_num_tr_datapoints)

    print(f"Num training data points used: {len(corpus_keys)} (from {len(corpus)})")

    # 2) Load a sentence transformer model
    model = SentenceTransformer(model_name)
    model = model.to(device)

    # 3) Process the corpus into embeddings
    corpus_texts = [corpus[doc_id]['text'] for doc_id in corpus_keys]
    corpus_embeddings = model.encode(
        corpus_texts,
        convert_to_numpy=False,
        show_progress_bar=True,
        batch_size=batch_size
    )
    # Convert list of tensors to a single tensor
    corpus_embeddings = torch.stack(corpus_embeddings)

    # 4) Unload the dataset and model from memory
    del corpus
    del model
    torch.cuda.empty_cache()

    # 5) Return the embeddings
    return corpus_embeddings


In [ ]:
def compute_knn_cosine(embeddings: torch.Tensor, k=5):
    # IndexFlatIP equals cosine search only when vectors are L2-normalized.
    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
    embeddings = embeddings.cpu().numpy().astype(np.float32)

    d = embeddings.shape[1]
    index = faiss.IndexFlatIP(d)
    index.add(embeddings)

    # Query k+1 then remove self-neighbor so top-k loss uses true neighbors (Eq.2).
    _, indices = index.search(embeddings, k + 1)

    row_ids = np.arange(embeddings.shape[0])[:, None]
    mask_not_self = indices != row_ids
    filtered = np.where(mask_not_self, indices, -1)

    knn_indices = []
    for row in filtered:
        row = row[row >= 0]
        if row.shape[0] < k:
            raise ValueError("KNN row has fewer than k non-self neighbors.")
        knn_indices.append(row[:k])

    return np.stack(knn_indices, axis=0)


### Matryoshka-Adaptor Losses

In [ ]:
def loss_rec(embeddings, transformed_embeddings):
    # implementing L_rec (Eq. 3)

    # Keep both sides on the same cosine scale to avoid magnitude mismatch.
    embeddings_normalized = torch.nn.functional.normalize(embeddings, p=2, dim=1)
    transformed_embeddings_normalized = torch.nn.functional.normalize(transformed_embeddings, p=2, dim=1)

    return torch.abs(embeddings_normalized - transformed_embeddings_normalized).mean()


In [ ]:
def loss_pair(embeddings, transformed_embeddings):
    # Pairwise loss should ignore diagonal self-similarity (always ~1 after normalization),
    # otherwise the diagonal dominates and makes optimization unstable.
    embeddings = torch.nn.functional.normalize(embeddings, p=2, dim=1)
    sim_original = embeddings @ embeddings.T
    upper_mask = torch.triu(torch.ones_like(sim_original, dtype=torch.bool), diagonal=1)

    per_m_losses = []
    for m in m_values:
        c_transformed_embeddings = transformed_embeddings[:, :m]
        c_transformed_embeddings = torch.nn.functional.normalize(c_transformed_embeddings, p=2, dim=1)

        sim_matryoshka = c_transformed_embeddings @ c_transformed_embeddings.T
        per_m_losses.append(torch.abs(sim_original - sim_matryoshka)[upper_mask].mean())

    # Average across m-values so this loss scale remains stable when changing m_values length.
    return torch.stack(per_m_losses).mean()


In [ ]:
def loss_knn(embeddings, knn_embeddings, transformed_embeddings, transformed_knn_embeddings):
    # All similarities are cosine, so we L2-normalize before dot-product (Eq.2).
    embeddings_norm = torch.nn.functional.normalize(embeddings, p=2, dim=1)
    knn_embeddings_norm = torch.nn.functional.normalize(knn_embeddings, p=2, dim=2)

    embeddings_similarities = torch.sum(embeddings_norm.unsqueeze(1) * knn_embeddings_norm, dim=-1)

    per_m_losses = []
    for m in m_values:
        c_transformed_embeddings = transformed_embeddings[:, :m]
        c_transformed_embeddings = torch.nn.functional.normalize(c_transformed_embeddings, p=2, dim=1)

        c_transformed_knn_embeddings = transformed_knn_embeddings[:, :, :m]
        c_transformed_knn_embeddings = torch.nn.functional.normalize(c_transformed_knn_embeddings, p=2, dim=2)

        c_transformed_embeddings_similarities = torch.sum(
            c_transformed_embeddings.unsqueeze(1) * c_transformed_knn_embeddings, dim=-1
        )
        per_m_losses.append(torch.abs(embeddings_similarities - c_transformed_embeddings_similarities).mean())

    # Multi-m average keeps the scale stable when m_values length changes.
    return torch.stack(per_m_losses).mean()


### Training

In [ ]:
def training_loop(embeddings, knn_indices, matryoshka_adaptor,
                  optimizer, batch_size, k, max_tr_iterations, patience,
                  lambda_rec=1.0, lambda_pair=1.0, lambda_knn=1.0,
                  log_every=50, grad_clip_norm=1.0):
    """
    Training loop with Eq.4 weighted objective, device-safe tensors, and detailed loss logging.
    """

    model_device = next(matryoshka_adaptor.parameters()).device
    embeddings = torch.nn.functional.normalize(embeddings.to(model_device), p=2, dim=1)
    knn_indices = knn_indices.to(model_device)

    N, D = embeddings.shape
    print(f"Start training -> m_values={m_values}, k={k}, lambdas=(rec={lambda_rec}, pair={lambda_pair}, knn={lambda_knn})")

    best_loss = float('inf')
    best_state_dict = copy.deepcopy(matryoshka_adaptor.state_dict())
    epochs_without_improvement = 0
    total_iterations = 0
    epoch = 0

    while total_iterations < max_tr_iterations:
        print(f"Epoch {epoch + 1}")
        indices = torch.randperm(N, device=model_device)

        epoch_loss = 0.0
        epoch_rec = 0.0
        epoch_pair = 0.0
        epoch_knn = 0.0
        steps_in_epoch = 0

        for i in range(0, N, batch_size):
            if total_iterations >= max_tr_iterations:
                print("Reached maximum number of iterations. Stopping training.")
                break

            batch_indices = indices[i:i + batch_size]
            batch_embeddings = embeddings[batch_indices]

            batch_knn_indices = knn_indices[batch_indices]
            if batch_knn_indices.shape[1] != k:
                raise ValueError(f"Expected batch_knn_indices second dim == k ({k}), got {batch_knn_indices.shape}")
            batch_knn_embeddings = embeddings[batch_knn_indices]

            optimizer.zero_grad()
            matryoshka_embeddings = matryoshka_adaptor(batch_embeddings)

            batch_knn_embeddings_flattened = batch_knn_embeddings.reshape(-1, D)
            batch_knn_embeddings_processed = matryoshka_adaptor(batch_knn_embeddings_flattened)
            batch_knn_embeddings_processed = batch_knn_embeddings_processed.reshape(batch_indices.shape[0], k, D)

            l_rec = loss_rec(batch_embeddings, matryoshka_embeddings)
            l_pair = loss_pair(batch_embeddings, matryoshka_embeddings)
            l_knn = loss_knn(batch_embeddings, batch_knn_embeddings, matryoshka_embeddings, batch_knn_embeddings_processed)
            loss = lambda_rec * l_rec + lambda_pair * l_pair + lambda_knn * l_knn

            if torch.isnan(loss) or torch.isinf(loss):
                raise RuntimeError("Loss became NaN/Inf. Please check normalization and learning rate.")

            loss.backward()
            if grad_clip_norm is not None and grad_clip_norm > 0:
                torch.nn.utils.clip_grad_norm_(matryoshka_adaptor.parameters(), grad_clip_norm)
            optimizer.step()

            epoch_loss += loss.item()
            epoch_rec += l_rec.item()
            epoch_pair += l_pair.item()
            epoch_knn += l_knn.item()
            steps_in_epoch += 1
            total_iterations += 1

            if total_iterations % log_every == 0:
                print(
                    f"iter={total_iterations} | total={loss.item():.4f} | rec={l_rec.item():.4f} | "
                    f"pair={l_pair.item():.4f} | knn={l_knn.item():.4f} | lambdas=({lambda_rec},{lambda_pair},{lambda_knn})"
                )

        if steps_in_epoch == 0:
            break

        epoch_loss /= steps_in_epoch
        epoch_rec /= steps_in_epoch
        epoch_pair /= steps_in_epoch
        epoch_knn /= steps_in_epoch

        print(
            f"End epoch {epoch + 1}: total={epoch_loss:.4f}, rec={epoch_rec:.4f}, "
            f"pair={epoch_pair:.4f}, knn={epoch_knn:.4f}"
        )

        if epoch_loss < best_loss:
            best_loss = epoch_loss
            best_state_dict = copy.deepcopy(matryoshka_adaptor.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            print(f"No improvement for {patience} consecutive epochs. Stopping training.")
            break

        epoch += 1

    # Always restore the best checkpoint seen during training.
    matryoshka_adaptor.load_state_dict(best_state_dict)
    print(f"Restored best adaptor checkpoint with loss={best_loss:.4f}")


In [ ]:
# Load + Preprocess training data
corpus_embeddings = load_and_preprocess_data(tr_dataset_name, base_encoder)
# normalize s.t. each embedding is of length 1 (note: they might already be, just making sure)
corpus_embeddings = torch.nn.functional.normalize(corpus_embeddings, p=2, dim=1)
print(f"corpus_embeddings.shape = {corpus_embeddings.shape}")

./datasets/scifact/scifact.zip:   0%|          | 0.00/2.69M [00:00<?, ?iB/s]

  0%|          | 0/5183 [00:00<?, ?it/s]

Num training data points : 5183


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

corpus_embeddings.shape = torch.Size([5183, 384])


In [ ]:
# Compute KNN
corpus_embeddings_knn_indices = compute_knn_cosine(corpus_embeddings, k=k)
print(f"corpus_embeddings_knn_indices.shape = {corpus_embeddings_knn_indices.shape}")
corpus_embeddings_knn_indices_torch = torch.from_numpy(corpus_embeddings_knn_indices)

corpus_embeddings_knn_indices.shape = (5183, 5)


In [ ]:
# Create the Matryoshka Adaptor
matryoshka_adaptor = MatryoshkaAdaptor(embedding_dim, ma_hidden_dim).to(device)
optimizer = optimizer_fn(matryoshka_adaptor.parameters())

In [ ]:
# Train the Adaptor
training_loop(
    corpus_embeddings,
    corpus_embeddings_knn_indices_torch,
    matryoshka_adaptor,
    optimizer,
    batch_size,
    k,
    max_tr_iterations,
    patience,
    lambda_rec=TRAIN_LAMBDA_REC,
    lambda_pair=TRAIN_LAMBDA_PAIR,
    lambda_knn=TRAIN_LAMBDA_KNN,
    log_every=log_every,
    grad_clip_norm=grad_clip_norm
)


Epoch 1
End of epoch 1: Loss = 12.49930044412613
Epoch 2
End of epoch 2: Loss = 9.98026739358902
Epoch 3
End of epoch 3: Loss = 8.894339954853058
Epoch 4
End of epoch 4: Loss = 8.156943577528
Epoch 5
End of epoch 5: Loss = 7.559378868341446
Epoch 6
End of epoch 6: Loss = 7.185621958971024
Epoch 7
End of epoch 7: Loss = 6.869057089090347
Epoch 8
End of epoch 8: Loss = 6.582199317216873
Epoch 9
End of epoch 9: Loss = 6.423422974348068
Epoch 10
End of epoch 10: Loss = 6.211178076267243
Epoch 11
End of epoch 11: Loss = 6.070136296749115
Epoch 12
End of epoch 12: Loss = 5.954926282167435
Epoch 13
End of epoch 13: Loss = 5.894255524873733
Epoch 14
End of epoch 14: Loss = 5.776445925235748
Epoch 15
End of epoch 15: Loss = 5.6798536539077755
Epoch 16
End of epoch 16: Loss = 5.590610152482986
Epoch 17
End of epoch 17: Loss = 5.611567181348801
Epoch 18
End of epoch 18: Loss = 5.528550541400909
Epoch 19
End of epoch 19: Loss = 5.438445997238159
Epoch 20
End of epoch 20: Loss = 5.365631812810898
E

### Save / Load the model

In [ ]:
# file_path = 'matryoshka_adaptor_scifact.pth'
# torch.save(matryoshka_adaptor.state_dict(), file_path)
# matryoshka_adaptor = MatryoshkaAdaptor(embedding_dim, ma_hidden_dim).to(device)
# matryoshka_adaptor.load_state_dict(torch.load(file_path))

## Sanity checks for Matryoshka property (unsupervised)
Các kiểm tra dưới đây xác minh rằng adaptor giúp giữ cấu trúc láng giềng/similarity tốt hơn truncate baseline ở dim nhỏ.


In [ ]:
# Build normalized original/adapted embeddings for sanity checks.
# Input: corpus_embeddings + trained adaptor. Output: 2 tensor (N, D) đã L2-normalize.
matryoshka_adaptor.eval()
corpus_embeddings_orig = torch.nn.functional.normalize(corpus_embeddings, p=2, dim=1).cpu()
with torch.no_grad():
    corpus_embeddings_adapted = matryoshka_adaptor(corpus_embeddings_orig.to(device))
    corpus_embeddings_adapted = torch.nn.functional.normalize(corpus_embeddings_adapted, p=2, dim=1).cpu()

print(f"Sanity tensors ready: original={corpus_embeddings_orig.shape}, adapted={corpus_embeddings_adapted.shape}")


In [ ]:
# Neighbor overlap@10 test
# overlap@10 = tỷ lệ phần tử chung giữa top-10 theo full-dim gold và top-10 theo dim m.
def faiss_topk_no_self(corpus_vecs: np.ndarray, query_vecs: np.ndarray, query_ids: np.ndarray, topk: int):
    idx = faiss.IndexFlatIP(corpus_vecs.shape[1])
    idx.add(corpus_vecs.astype(np.float32))
    _, nn = idx.search(query_vecs.astype(np.float32), topk + 1)

    results = []
    for row, qid in zip(nn, query_ids):
        row = row[row != qid]
        results.append(row[:topk])
    return np.stack(results, axis=0)

N = corpus_embeddings_orig.shape[0]
sample_size = min(200, N)
sample_ids = np.array(python_random.sample(range(N), sample_size))

orig_np = corpus_embeddings_orig.numpy()
adapt_np = corpus_embeddings_adapted.numpy()
gold_neighbors = faiss_topk_no_self(orig_np, orig_np[sample_ids], sample_ids, topk=10)

print("\nOverlap@10 (higher is better)")
print("m	baseline_truncate	adaptor")
for m in m_values:
    base_m = orig_np[:, :m]
    base_m = base_m / (np.linalg.norm(base_m, axis=1, keepdims=True) + 1e-12)
    adapt_m = adapt_np[:, :m]
    adapt_m = adapt_m / (np.linalg.norm(adapt_m, axis=1, keepdims=True) + 1e-12)

    base_neighbors = faiss_topk_no_self(base_m, base_m[sample_ids], sample_ids, topk=10)
    adapt_neighbors = faiss_topk_no_self(adapt_m, adapt_m[sample_ids], sample_ids, topk=10)

    base_overlap = np.mean([len(set(g).intersection(set(b))) / 10.0 for g, b in zip(gold_neighbors, base_neighbors)])
    adapt_overlap = np.mean([len(set(g).intersection(set(a))) / 10.0 for g, a in zip(gold_neighbors, adapt_neighbors)])

    print(f"{m}\t{base_overlap:.4f}\t\t{adapt_overlap:.4f}")


In [ ]:
# Cosine correlation + MSE test
# Mục tiêu: cosine ở prefix m sau adaptor gần cosine full-dim gốc hơn baseline truncate.
num_pairs = min(4000, N * 8)
pair_i = np.array([python_random.randrange(N) for _ in range(num_pairs)])
pair_j = np.array([python_random.randrange(N) for _ in range(num_pairs)])
mask_same = pair_i == pair_j
while mask_same.any():
    pair_j[mask_same] = np.array([python_random.randrange(N) for _ in range(mask_same.sum())])
    mask_same = pair_i == pair_j

cos_full = np.sum(orig_np[pair_i] * orig_np[pair_j], axis=1)

print("\nCosine similarity sanity check")
print("m	pearson_base	pearson_adaptor	mse_base	mse_adaptor")
for m in m_values:
    base_i = orig_np[pair_i, :m]
    base_j = orig_np[pair_j, :m]
    base_i = base_i / (np.linalg.norm(base_i, axis=1, keepdims=True) + 1e-12)
    base_j = base_j / (np.linalg.norm(base_j, axis=1, keepdims=True) + 1e-12)
    cos_base = np.sum(base_i * base_j, axis=1)

    adapt_i = adapt_np[pair_i, :m]
    adapt_j = adapt_np[pair_j, :m]
    adapt_i = adapt_i / (np.linalg.norm(adapt_i, axis=1, keepdims=True) + 1e-12)
    adapt_j = adapt_j / (np.linalg.norm(adapt_j, axis=1, keepdims=True) + 1e-12)
    cos_adapt = np.sum(adapt_i * adapt_j, axis=1)

    pearson_base = np.corrcoef(cos_full, cos_base)[0, 1]
    pearson_adapt = np.corrcoef(cos_full, cos_adapt)[0, 1]
    mse_base = np.mean((cos_full - cos_base) ** 2)
    mse_adapt = np.mean((cos_full - cos_adapt) ** 2)

    print(f"{m}\t{pearson_base:.4f}\t	{pearson_adapt:.4f}\t	{mse_base:.6f}\t{mse_adapt:.6f}")

# Optional norm stability summary (reg + skip connection should keep norms controlled).
print("\nNorm summary")
print(f"orig norm mean/std: {np.linalg.norm(orig_np, axis=1).mean():.4f}/{np.linalg.norm(orig_np, axis=1).std():.4f}")
print(f"adapt norm mean/std: {np.linalg.norm(adapt_np, axis=1).mean():.4f}/{np.linalg.norm(adapt_np, axis=1).std():.4f}")


# Evaluations

In [ ]:
class SentenceTransformerwMatryoskaAdaptor(SentenceTransformer):
    def __init__(self, base_model_name: str, adaptor: torch.nn.Module, device=None):
        super().__init__(base_model_name)
        # We keep adaptor on a known target device and avoid overriding SentenceTransformer.device property.
        self.adaptor_device = torch.device(device if device is not None else ("cuda" if torch.cuda.is_available() else "cpu"))
        self.adaptor = adaptor.to(self.adaptor_device).eval()

    def encode(self, sentences, **kwargs):
        # We override encode() so every evaluation call uses the same pipeline:
        # base(full dim) -> L2 normalize -> adaptor -> L2 normalize -> truncate[:m].
        # Truncation must happen AFTER adaptor to preserve Matryoshka behavior learned across full-dim context.
        m_value = getattr(self, 'truncate_dim', None)

        restore_truncate_dim = self.truncate_dim if hasattr(self, 'truncate_dim') else None
        return_numpy = kwargs.pop('convert_to_numpy', True)
        return_tensor = kwargs.pop('convert_to_tensor', False)

        try:
            self.truncate_dim = None
            kwargs['convert_to_tensor'] = True
            kwargs['convert_to_numpy'] = False

            with torch.no_grad():
                base_embeds = super().encode(sentences, **kwargs)
                if not isinstance(base_embeds, torch.Tensor):
                    base_embeds = torch.tensor(base_embeds)

                base_embeds = base_embeds.to(self.adaptor_device)
                base_embeds = torch.nn.functional.normalize(base_embeds, p=2, dim=1)

                adapted_embeds = self.adaptor(base_embeds)
                adapted_embeds = torch.nn.functional.normalize(adapted_embeds, p=2, dim=1)

                if m_value is not None:
                    adapted_embeds = adapted_embeds[:, :m_value]
                    adapted_embeds = torch.nn.functional.normalize(adapted_embeds, p=2, dim=1)

            if return_tensor:
                return adapted_embeds
            if return_numpy:
                return adapted_embeds.cpu().numpy()
            return adapted_embeds.cpu().tolist()
        finally:
            self.truncate_dim = restore_truncate_dim

    # Compatibility aliases used by some MTEB/BEIR versions.
    def encode_queries(self, queries, **kwargs):
        return self.encode(queries, **kwargs)

    def encode_corpus(self, corpus, **kwargs):
        if isinstance(corpus, list) and len(corpus) > 0 and isinstance(corpus[0], dict):
            texts = [f"{doc.get('title', '')} {doc.get('text', '')}".strip() for doc in corpus]
            return self.encode(texts, **kwargs)
        return self.encode(corpus, **kwargs)


### EVAL CONFIG


In [ ]:
# Evaluation config kept explicit so new users can tweak only this block.
EVAL_DATASETS = ["scifact"]  # Colab-friendly default (paper-style retrieval eval on SciFact)
EVAL_M_VALUES = m_values
METRIC = "ndcg_at_10"  # corresponds to nDCG@10 in MTEB result keys
MAX_EVAL_QUERIES = None  # Optional: set an int only when you must cap eval for timeout/debug

print(f"Eval datasets: {EVAL_DATASETS}")
print(f"m values: {EVAL_M_VALUES}")
print(f"Metric key: {METRIC}")


In [ ]:
from pathlib import Path
import json
import pandas as pd

MTEB_TASK_NAME_MAP = {
    "scifact": "SciFact",
    "msmarco": "MSMARCO",
}


def _extract_metric(eval_output, metric_key):
    # MTEB returns nested objects; this helper safely extracts test nDCG@10.
    return eval_output[0].scores["test"][0][metric_key]


def run_mteb_eval_milestone3(base_model_name, adaptor_module, datasets, m_values, metric_key):
    """Run baseline-vs-adaptor retrieval eval for every dataset and truncation m."""
    results = {}

    for dataset in datasets:
        tasks = mteb.get_tasks(tasks=[MTEB_TASK_NAME_MAP[dataset]])
        benchmark = MTEB(tasks=tasks)

        baseline_model = SentenceTransformer(base_model_name)
        adaptor_model = SentenceTransformerwMatryoskaAdaptor(base_model_name, adaptor=adaptor_module)

        dataset_results = {"baseline": {}, "adaptor": {}, "delta": {}}
        print(f"\nRunning dataset: {dataset}")

        for m in m_values:
            # Baseline: original embedding then truncate[:m] in SentenceTransformer.
            baseline_model.truncate_dim = m
            baseline_score = _extract_metric(benchmark.run(baseline_model, output_folder=None), metric_key)

            # Adaptor: wrapper applies adaptor first, then truncate[:m] inside overridden encode().
            adaptor_model.truncate_dim = m
            adaptor_score = _extract_metric(benchmark.run(adaptor_model, output_folder=None), metric_key)

            dataset_results["baseline"][m] = baseline_score
            dataset_results["adaptor"][m] = adaptor_score
            dataset_results["delta"][m] = adaptor_score - baseline_score
            print(f"  m={m:<4} baseline={baseline_score:.5f} adaptor={adaptor_score:.5f} delta={adaptor_score - baseline_score:+.5f}")

        del baseline_model
        del adaptor_model
        results[dataset] = dataset_results

    return results


In [ ]:
# Run Milestone-3 evaluation loop and keep outputs in one canonical dict.
results = run_mteb_eval_milestone3(
    base_model_name=base_encoder,
    adaptor_module=matryoshka_adaptor,
    datasets=EVAL_DATASETS,
    m_values=EVAL_M_VALUES,
    metric_key=METRIC,
)

# Validation check: adaptor should differ from baseline for at least one m.
for dataset, dataset_results in results.items():
    deltas = [dataset_results["delta"][m] for m in EVAL_M_VALUES]
    has_difference = any(abs(d) > 1e-12 for d in deltas)
    print(f"Validation ({dataset}): baseline vs adaptor differs at least one m -> {has_difference}")


In [ ]:
# Show a clear result table per dataset: baseline, adaptor, and delta by m.
result_tables = {}
for dataset, dataset_results in results.items():
    table = pd.DataFrame({
        "m": EVAL_M_VALUES,
        "baseline": [dataset_results["baseline"][m] for m in EVAL_M_VALUES],
        "adaptor": [dataset_results["adaptor"][m] for m in EVAL_M_VALUES],
        "delta": [dataset_results["delta"][m] for m in EVAL_M_VALUES],
    })
    result_tables[dataset] = table
    print(f"\nResults table - {dataset} ({METRIC})")
    display(table)


In [ ]:
# Plot m vs nDCG@10 (baseline vs adaptor) for each dataset.
for dataset, table in result_tables.items():
    plt.figure(figsize=(8, 4.8))
    plt.plot(table["m"], table["baseline"], marker="o", label="Baseline (truncate)")
    plt.plot(table["m"], table["adaptor"], marker="o", label="Matryoshka Adaptor")
    plt.title(f"m vs nDCG@10 on {dataset}")
    plt.xlabel("Embedding dimensions (m)")
    plt.ylabel("nDCG@10")
    plt.grid(True, alpha=0.25)
    plt.legend()
    plt.show()


In [ ]:
# Save raw results so runs are reproducible and easy to compare later.
results_path = Path("results_mteb_eval.json")
serializable_results = {
    dataset: {
        section: {str(m): score for m, score in values.items()}
        for section, values in dataset_results.items()
    }
    for dataset, dataset_results in results.items()
}

with open(results_path, "w", encoding="utf-8") as f:
    json.dump(serializable_results, f, indent=2)

print(f"Saved results to: {results_path.resolve()}")
